# Agent 1 Structure Evaluation

This notebook evaluates Agent 1 structured representations.

## Current design

| Component | Matching strategy |
|---|---|
| participants | set precision / recall / F1 |
| speech_act | strict raw-tag matching + dictionary-based relaxed category matching |
| intended_meaning | soft many-to-many matching with BERTScore |
| actor / recipient | exact normalized string match |
| action / object | BERTScore similarity |
| semantic tuples | one-to-one Hungarian matching using weighted slot score |
| final_outcome | NLI entailment similarity |



In [1]:

!pip install -q bert-score "transformers>=4.40"

import json
import re
from pathlib import Path
from difflib import SequenceMatcher

import numpy as np
import pandas as pd
from scipy.optimize import linear_sum_assignment

try:
    from bert_score import BERTScorer
    HAS_BERTSCORE = True
except Exception:
    HAS_BERTSCORE = False

try:
    from transformers import pipeline
    HAS_TRANSFORMERS = True
except Exception:
    HAS_TRANSFORMERS = False

HAS_BERTSCORE, HAS_TRANSFORMERS


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.2 MB/s eta 0:00:00


(True, True)

## 1. Example Data

In [2]:
gold_data = [
    {
        "test_index": 23,
        "participants": ["Anne", "Irene", "Jane"],
        "semantic_grounding": [
            {
                "event_id": 1,
                "speaker": "Anne",
                "speech_act": "disclosure",
                "intended_meaning": "Anne confirms that Mark lied to her about his age.",
                "actor": "Mark",
                "action": "lied about age",
                "object": "age",
                "recipient": "Anne",
                "evidence": ["he told me he's 30", "he's 40"]
            },
            {
                "event_id": 2,
                "speaker": "Jane",
                "speech_act": "reaction",
                "intended_meaning": "Jane reacts with surprise to Mark's real age.",
                "actor": "Jane",
                "action": "reacts with surprise",
                "object": "Mark's real age",
                "recipient": "Anne",
                "evidence": ["No way, he's 40?"]
            }
        ],
        "final_outcome": "Mark lied to Anne about his age and is actually 40."
    }
]

pred_data = [
    {
        "test_index": 23,
        "participants": ["Anne", "Irene", "Jane"],
        "semantic_grounding": [
            {
                "event_id": 1,
                "speaker": "Anne",
                "speech_act": "statement",
                "intended_meaning": "Anne reveals that Mark was dishonest about being 30 because he is actually 40, and Jane is surprised.",
                "actor": "Mark",
                "action": "was dishonest about his age",
                "object": "age",
                "recipient": "Anne",
                "evidence": ["he told me he's 30", "he's 40"]
            }
        ],
        "final_outcome": "Mark was dishonest with Anne about his age; he is 40, not 30."
    }
]


## 2. Load Files

In [5]:
def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)[0:3]

def load_jsonl(path):
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                data.append(json.loads(line))
    return data[0:3]

def save_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

gold_raw = load_json("agent1_semantic_representation_gpt5.5.json")
pred_raw = load_jsonl("semantic_qwen27b_50samples_intermediate.jsonl")


## 3. Normalization

Two separate functions (see header note):

- `normalize_entity` — aggressive, for exact-match slots.
- `normalize_semantic` — light, for NLI / BERTScore inputs (keeps case and
  sentence punctuation).


In [6]:
def normalize_entity(x):
    """Aggressive normalization for exact-match slots.

    Used for: actor, recipient, participants, speaker, raw speech_act tags.
    Lowercases and strips punctuation so that "Mark." and "mark" compare equal.
    """
    if x is None:
        return None
    x = str(x).strip().lower()
    x = re.sub(r"[^\w\s\-']", "", x)
    x = re.sub(r"\s+", " ", x)
    return x if x else None

def normalize_semantic(x):
    """Light normalization for text sent to NLI / BERTScore models.

    Used for: intended_meaning, final_outcome, action, object.
    Keeps case and sentence punctuation (NLI and BERTScore models are
    case- and punctuation-sensitive); only collapses whitespace.
    """
    if x is None:
        return None
    x = str(x).strip()
    x = re.sub(r"\s+", " ", x)
    return x if x else None

def normalize_text(x):
    return normalize_entity(x)

def normalize_action(x):
    return normalize_semantic(x)

def normalize_object(x):
    return normalize_semantic(x)

def safe_list(x):
    if x is None:
        return []
    if isinstance(x, list):
        return x
    return [x]


## 4. Flatten Structures

In [7]:
def get_test_id(dialogue):
    return dialogue.get("test_index")

def get_groundings(dialogue):
    return safe_list(dialogue.get("semantic_grounding", []))

def flatten_participants(dialogue):
    # Participants are exact-match entities.
    return [normalize_entity(p) for p in safe_list(dialogue.get("participants", []))
            if normalize_entity(p)]

def flatten_intended_meanings(dialogue):
    # Intended meanings are full sentences -> semantic normalization.
    meanings = []
    for item in get_groundings(dialogue):
        if isinstance(item, dict):
            meaning = normalize_semantic(item.get("intended_meaning"))
            if meaning:
                meanings.append(meaning)
    return meanings

def flatten_speech_act_items(dialogue):
    items = []
    for item in get_groundings(dialogue):
        if isinstance(item, dict):
            # Speech-act tag is an exact-match label.
            tag = normalize_entity(item.get("speech_act"))
            if tag:
                items.append({
                    "speech_act": tag,
                    # Keep the raw meaning for inspection / debugging.
                    "intended_meaning": item.get("intended_meaning"),
                })
    return items

def flatten_speech_acts(dialogue):
    return [x["speech_act"] for x in flatten_speech_act_items(dialogue)]

def flatten_semantic_tuples(dialogue):
    """Return tuples in the format: (actor, action, object, recipient).
    """
    tuples = []
    for item in get_groundings(dialogue):
        if not isinstance(item, dict):
            continue
        tup = (
            normalize_entity(item.get("actor")),
            normalize_semantic(item.get("action")),
            normalize_semantic(item.get("object")),
            normalize_entity(item.get("recipient")),
        )
        if any(v is not None for v in tup):
            tuples.append(tup)
    return tuples

print("Gold meanings:", flatten_intended_meanings(gold_data[0]))
print("Pred meanings:", flatten_intended_meanings(pred_data[0]))
print("Gold tuples:", flatten_semantic_tuples(gold_data[0]))
print("Pred tuples:", flatten_semantic_tuples(pred_data[0]))


Gold meanings: ['Anne confirms that Mark lied to her about his age.', "Jane reacts with surprise to Mark's real age."]
Pred meanings: ['Anne reveals that Mark was dishonest about being 30 because he is actually 40, and Jane is surprised.']
Gold tuples: [('mark', 'lied about age', 'age', 'anne'), ('jane', 'reacts with surprise', "Mark's real age", 'anne')]
Pred tuples: [('mark', 'was dishonest about his age', 'age', 'anne')]


## 5. BERTScore Similarity

BERTScore is used for:

- `action`
- `object`


Implementation note: the `BERTScorer` model is loaded once and reused. Pairs are
scored in **batches** via `bertscore_similarity_batch`, which is far faster on
GPU than scoring one pair at a time. Individual-pair results are cached, and
`bertscore_similarity` (single pair) is kept for convenience.

Inputs are normalized with `normalize_semantic` (case and punctuation kept), not
the aggressive entity normalizer.


In [8]:
BERTSCORE_MODEL = "roberta-large"
BERTSCORE_LANG = "en"
BERTSCORE_RESCALE_WITH_BASELINE = True   # rescale against baseline
BERTSCORE_BATCH_SIZE = 64

_BERTSCORE_CACHE = {}
_BERTSCORER = None

def get_bertscorer():
    """Load BERTScorer once and reuse it."""
    global _BERTSCORER
    if _BERTSCORER is None:
        if not HAS_BERTSCORE:
            return None
        print(f"Loading BERTScorer once: {BERTSCORE_MODEL}")
        _BERTSCORER = BERTScorer(
            model_type=BERTSCORE_MODEL,
            lang=BERTSCORE_LANG,
            rescale_with_baseline=BERTSCORE_RESCALE_WITH_BASELINE,
        )
    return _BERTSCORER

def string_similarity(a, b):
    a = normalize_semantic(a)
    b = normalize_semantic(b)
    if a is None and b is None:
        return 1.0
    if a is None or b is None:
        return 0.0
    if a == b:
        return 1.0
    return SequenceMatcher(None, a, b).ratio()

def _bertscore_cache_key(a, b):
    return (a, b, BERTSCORE_MODEL, BERTSCORE_LANG, BERTSCORE_RESCALE_WITH_BASELINE)

def _trivial_bertscore(a, b):
    """Return a score for trivial pairs, else None (needs the model)."""
    if a is None and b is None:
        return 1.0
    if a is None or b is None:
        return 0.0
    if a == b:
        return 1.0
    return None

def bertscore_similarity_batch(pairs):
    """Score many (a, b) pairs at once. Returns a list of F1 floats.

    Trivial pairs (None / identical) are resolved without the model. All
    remaining unique pairs are scored in a single batched BERTScorer call,
    which is dramatically faster on GPU than per-pair scoring.
    """
    norm_pairs = [(normalize_semantic(a), normalize_semantic(b)) for a, b in pairs]
    results = [None] * len(norm_pairs)

    to_score = {}  # cache_key -> (a, b)
    for i, (a, b) in enumerate(norm_pairs):
        triv = _trivial_bertscore(a, b)
        if triv is not None:
            results[i] = triv
            continue
        key = _bertscore_cache_key(a, b)
        if key in _BERTSCORE_CACHE:
            results[i] = _BERTSCORE_CACHE[key]
        else:
            to_score[key] = (a, b)

    if to_score:
        scorer = get_bertscorer()
        keys = list(to_score.keys())
        cands = [to_score[k][0] for k in keys]
        refs = [to_score[k][1] for k in keys]
        if scorer is None:
            for k in keys:
                a, b = to_score[k]
                _BERTSCORE_CACHE[k] = string_similarity(a, b)
        else:
            try:
                P, R, F1 = scorer.score(cands, refs, batch_size=BERTSCORE_BATCH_SIZE)
                for k, f in zip(keys, F1):
                    _BERTSCORE_CACHE[k] = float(f.item())
            except Exception as e:
                print("BERTScorer batch failed; falling back to string similarity:", repr(e))
                for k in keys:
                    a, b = to_score[k]
                    _BERTSCORE_CACHE[k] = string_similarity(a, b)

    # Fill in any results still missing (were just scored).
    for i, (a, b) in enumerate(norm_pairs):
        if results[i] is None:
            results[i] = _BERTSCORE_CACHE[_bertscore_cache_key(a, b)]
    return results

def bertscore_similarity(a, b):
    """Score a single (a, b) pair. Convenience wrapper over the batch path."""
    return bertscore_similarity_batch([(a, b)])[0]

print("BERTScore available:", HAS_BERTSCORE)
print("BERTScorer will load lazily on first scoring call.")


BERTScore available: True
BERTScorer will load lazily on first scoring call.


## 6. NLI Similarity

NLI is used for:

- `final_outcome`

Implementation note: pairs are scored in **batches** via `nli_entailment_prob_batch`,
which sends all premise/hypothesis pairs to the pipeline in one GPU call.
`evaluate_intended_meanings` prewarms the full bidirectional NLI grid, and
`evaluate_final_outcome` prewarms both directions. Per-pair results are cached in
`_NLI_CACHE`, so `nli_entailment_prob` (single pair) just reads the warm cache.

Inputs use `normalize_semantic` (case and punctuation kept).

In [9]:
NLI_MODEL = "cross-encoder/nli-deberta-v3-base"
NLI_BATCH_SIZE = 32

_NLI_PIPE = None
_NLI_CACHE = {}

def get_nli_pipeline():
    global _NLI_PIPE
    if _NLI_PIPE is None:
        if not HAS_TRANSFORMERS:
            return None
        # top_k=None replaces the deprecated return_all_scores=True.
        _NLI_PIPE = pipeline("text-classification", model=NLI_MODEL, top_k=None)
    return _NLI_PIPE

def _extract_entailment_score(output):
    # The pipeline returns a list of label dicts; for a single input it may be
    # wrapped one level deep.
    if isinstance(output, list) and len(output) == 1 and isinstance(output[0], list):
        output = output[0]
    best = 0.0
    for item in output:
        label = str(item.get("label", "")).lower()
        if "entail" in label:
            best = max(best, float(item.get("score", 0.0)))
    return best

def _nli_cache_key(premise, hypothesis):
    return (premise, hypothesis, NLI_MODEL)

def _trivial_nli(premise, hypothesis):
    """Return a score for trivial pairs, else None (needs the model)."""
    if premise is None and hypothesis is None:
        return 1.0
    if premise is None or hypothesis is None:
        return 0.0
    if premise == hypothesis:
        return 1.0
    return None

def nli_entailment_prob_batch(pairs):
    """Score many (premise, hypothesis) pairs at once. Returns a list of floats.

    Trivial pairs (None / identical) are resolved without the model. All
    remaining unique pairs are scored in one batched pipeline call, which is far
    faster on GPU than per-pair scoring. Results populate _NLI_CACHE.
    """
    norm_pairs = [(normalize_semantic(p), normalize_semantic(h)) for p, h in pairs]
    results = [None] * len(norm_pairs)

    to_score = {}  # cache_key -> (premise, hypothesis)
    for i, (p, h) in enumerate(norm_pairs):
        triv = _trivial_nli(p, h)
        if triv is not None:
            results[i] = triv
            continue
        key = _nli_cache_key(p, h)
        if key in _NLI_CACHE:
            results[i] = _NLI_CACHE[key]
        else:
            to_score[key] = (p, h)

    if to_score:
        nli = get_nli_pipeline()
        keys = list(to_score.keys())
        if nli is None:
            for k in keys:
                p, h = to_score[k]
                _NLI_CACHE[k] = string_similarity(p, h)
        else:
            inputs = [{"text": to_score[k][0], "text_pair": to_score[k][1]} for k in keys]
            try:
                outputs = nli(inputs, batch_size=NLI_BATCH_SIZE)
                # With a list input the pipeline returns one result per item.
                for k, out in zip(keys, outputs):
                    _NLI_CACHE[k] = _extract_entailment_score(out)
            except Exception as e:
                print("NLI batch failed; falling back to string similarity:", repr(e))
                for k in keys:
                    p, h = to_score[k]
                    _NLI_CACHE[k] = string_similarity(p, h)

    for i, (p, h) in enumerate(norm_pairs):
        if results[i] is None:
            results[i] = _NLI_CACHE[_nli_cache_key(p, h)]
    return results

def nli_entailment_prob(premise, hypothesis):
    """Score a single (premise, hypothesis) pair. Wrapper over the batch path."""
    return nli_entailment_prob_batch([(premise, hypothesis)])[0]

def bidirectional_nli_similarity(a, b):
    forward = nli_entailment_prob(a, b)
    backward = nli_entailment_prob(b, a)
    return (forward + backward) / 2

print("Transformers available:", HAS_TRANSFORMERS)


Transformers available: True


## 7. Matching Functions

In [10]:
def soft_many_to_many_match(pred_items, gold_items, sim_func, threshold=None):
    n_pred = len(pred_items)
    n_gold = len(gold_items)

    if n_pred == 0 and n_gold == 0:
        return {
            "precision": 1.0,
            "recall": 1.0,
            "f1": 1.0,
            "pred_best_matches": [],
            "gold_best_matches": [],
            "similarity_matrix": np.zeros((0, 0)),
        }

    if n_pred == 0 or n_gold == 0:
        return {
            "precision": 0.0,
            "recall": 0.0,
            "f1": 0.0,
            "pred_best_matches": [],
            "gold_best_matches": [],
            "similarity_matrix": np.zeros((n_pred, n_gold)),
        }

    sim = np.array([[sim_func(p, g) for g in gold_items] for p in pred_items])

    pred_best_scores = sim.max(axis=1)
    pred_best_gold_idx = sim.argmax(axis=1)

    gold_best_scores = sim.max(axis=0)
    gold_best_pred_idx = sim.argmax(axis=0)

    if threshold is not None:
        pred_scores_for_precision = np.where(pred_best_scores >= threshold, pred_best_scores, 0.0)
        gold_scores_for_recall = np.where(gold_best_scores >= threshold, gold_best_scores, 0.0)
    else:
        pred_scores_for_precision = pred_best_scores
        gold_scores_for_recall = gold_best_scores

    precision = float(np.mean(pred_scores_for_precision)) if n_pred else 0.0
    recall = float(np.mean(gold_scores_for_recall)) if n_gold else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0

    pred_best_matches = [
        {
            "pred_index": int(i),
            "gold_index": int(pred_best_gold_idx[i]),
            "pred_item": pred_items[i],
            "gold_item": gold_items[int(pred_best_gold_idx[i])],
            "similarity": float(pred_best_scores[i]),
        }
        for i in range(n_pred)
    ]

    gold_best_matches = [
        {
            "gold_index": int(j),
            "pred_index": int(gold_best_pred_idx[j]),
            "gold_item": gold_items[j],
            "pred_item": pred_items[int(gold_best_pred_idx[j])],
            "similarity": float(gold_best_scores[j]),
        }
        for j in range(n_gold)
    ]

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "pred_best_matches": pred_best_matches,
        "gold_best_matches": gold_best_matches,
        "similarity_matrix": sim,
    }

def bipartite_match(pred_items, gold_items, sim_func, threshold=0.70):
    n_pred = len(pred_items)
    n_gold = len(gold_items)

    if n_pred == 0 or n_gold == 0:
        both_empty = n_pred == 0 and n_gold == 0
        return {
            "matches": [],
            "similarity_matrix": np.zeros((n_pred, n_gold)),
            "precision": 1.0 if both_empty else 0.0,
            "recall": 1.0 if both_empty else 0.0,
            "f1": 1.0 if both_empty else 0.0,
            "avg_matched_similarity": 0.0,
        }

    sim = np.array([[sim_func(p, g) for g in gold_items] for p in pred_items])
    cost = 1 - sim
    pred_idx, gold_idx = linear_sum_assignment(cost)

    matches = []
    for p_i, g_i in zip(pred_idx, gold_idx):
        score = sim[p_i, g_i]
        if score >= threshold:
            matches.append({
                "pred_index": int(p_i),
                "gold_index": int(g_i),
                "pred_item": pred_items[p_i],
                "gold_item": gold_items[g_i],
                "similarity": float(score),
            })

    precision = len(matches) / n_pred if n_pred else 0.0
    recall = len(matches) / n_gold if n_gold else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    avg_sim = float(np.mean([m["similarity"] for m in matches])) if matches else 0.0

    return {
        "matches": matches,
        "similarity_matrix": sim,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "avg_matched_similarity": avg_sim,
    }


## 8. Participant Evaluation

In [11]:
def evaluate_participants(pred_dialogue, gold_dialogue):
    pred = set(flatten_participants(pred_dialogue))
    gold = set(flatten_participants(gold_dialogue))
    tp = len(pred & gold)
    precision = tp / len(pred) if pred else (1.0 if not gold else 0.0)
    recall = tp / len(gold) if gold else (1.0 if not pred else 0.0)
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {
        "participant_precision": precision,
        "participant_recall": recall,
        "participant_f1": f1,
        "participant_exact_match": pred == gold,
    }

evaluate_participants(pred_data[0], gold_data[0])


{'participant_precision': 1.0,
 'participant_recall': 1.0,
 'participant_f1': 1.0,
 'participant_exact_match': True}

## 9. Intended Meaning Evaluation (BERTScore)

`intended_meaning` is evaluated with **BERTScore only**. Each predicted meaning
is compared against each gold meaning to form an `n_pred x n_gold` similarity
matrix (one batched, rescaled BERTScore call), then scored with soft
many-to-many matching:

- precision = mean of each prediction's best similarity to any gold meaning;
- recall = mean of each gold meaning's best similarity to any prediction;
- F1 = harmonic mean of the two.


In [12]:
def _matrix_match_from_sim(pred_items, gold_items, sim, threshold=None):
    """Soft many-to-many precision/recall/F1 from a precomputed similarity matrix."""
    n_pred, n_gold = len(pred_items), len(gold_items)

    if n_pred == 0 and n_gold == 0:
        return {"precision": 1.0, "recall": 1.0, "f1": 1.0,
                "pred_best_matches": [], "gold_best_matches": [],
                "similarity_matrix": np.zeros((0, 0))}
    if n_pred == 0 or n_gold == 0:
        return {"precision": 0.0, "recall": 0.0, "f1": 0.0,
                "pred_best_matches": [], "gold_best_matches": [],
                "similarity_matrix": np.zeros((n_pred, n_gold))}

    pred_best = sim.max(axis=1)
    gold_best = sim.max(axis=0)
    pred_gold_idx = sim.argmax(axis=1)
    gold_pred_idx = sim.argmax(axis=0)

    if threshold is not None:
        pred_scored = np.where(pred_best >= threshold, pred_best, 0.0)
        gold_scored = np.where(gold_best >= threshold, gold_best, 0.0)
    else:
        pred_scored, gold_scored = pred_best, gold_best

    precision = float(np.mean(pred_scored))
    recall = float(np.mean(gold_scored))
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0

    pred_best_matches = [
        {"pred_index": int(i), "gold_index": int(pred_gold_idx[i]),
         "pred_item": pred_items[i], "gold_item": gold_items[int(pred_gold_idx[i])],
         "similarity": float(pred_best[i])}
        for i in range(n_pred)
    ]
    gold_best_matches = [
        {"gold_index": int(j), "pred_index": int(gold_pred_idx[j]),
         "gold_item": gold_items[j], "pred_item": pred_items[int(gold_pred_idx[j])],
         "similarity": float(gold_best[j])}
        for j in range(n_gold)
    ]
    return {"precision": precision, "recall": recall, "f1": f1,
            "pred_best_matches": pred_best_matches,
            "gold_best_matches": gold_best_matches,
            "similarity_matrix": sim}

def evaluate_intended_meanings(pred_dialogue, gold_dialogue, threshold=None):
    """Evaluate intended_meaning with BERTScore only."""
    pred_meanings = flatten_intended_meanings(pred_dialogue)
    gold_meanings = flatten_intended_meanings(gold_dialogue)
    n_pred, n_gold = len(pred_meanings), len(gold_meanings)

    if n_pred == 0 or n_gold == 0:
        sim = np.zeros((n_pred, n_gold))
    else:
        # One batched, rescaled BERTScore call for the whole grid.
        pairs = [(p, g) for p in pred_meanings for g in gold_meanings]
        sim = np.array(bertscore_similarity_batch(pairs)).reshape(n_pred, n_gold)

    result = _matrix_match_from_sim(pred_meanings, gold_meanings, sim, threshold)
    return {"n_pred_meanings": n_pred, "n_gold_meanings": n_gold,
            "bertscore": result}

meaning_eval = evaluate_intended_meanings(pred_data[0], gold_data[0])
print("BERTScore:", meaning_eval["bertscore"]["precision"],
      meaning_eval["bertscore"]["recall"], meaning_eval["bertscore"]["f1"])


Loading BERTScorer once: roberta-large


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERTScore: 0.5682100653648376 0.5247229337692261 0.5456013364619228


## 10. Semantic Tuple Evaluation

In [13]:
TUPLE_WEIGHTS = {
    "actor": 0.35,
    "action": 0.35,
    "object": 0.20,
    "recipient": 0.10,
}

def exact_entity_match(a, b):
    a = normalize_entity(a)
    b = normalize_entity(b)
    if a is None and b is None:
        return 1.0
    if a is None or b is None:
        return 0.0
    return 1.0 if a == b else 0.0

def tuple_similarity(pred_tuple, gold_tuple, weights=TUPLE_WEIGHTS):
    """Weighted slot similarity for a single tuple pair.

    Kept for convenience / single-pair use. The dataset path uses
    evaluate_tuples, which pre-batches all action/object BERTScore calls so the
    per-pair _BERTSCORE_CACHE is already warm by the time this runs.
    """
    p_actor, p_action, p_object, p_recipient = pred_tuple
    g_actor, g_action, g_object, g_recipient = gold_tuple

    scores = {
        "actor": exact_entity_match(p_actor, g_actor),
        "action": bertscore_similarity(p_action, g_action),
        "object": bertscore_similarity(p_object, g_object),
        "recipient": exact_entity_match(p_recipient, g_recipient),
    }
    return float(sum(weights[k] * scores[k] for k in weights))

def tuple_slot_scores(pred_tuple, gold_tuple):
    p_actor, p_action, p_object, p_recipient = pred_tuple
    g_actor, g_action, g_object, g_recipient = gold_tuple
    return {
        "actor": exact_entity_match(p_actor, g_actor),
        "action": bertscore_similarity(p_action, g_action),
        "object": bertscore_similarity(p_object, g_object),
        "recipient": exact_entity_match(p_recipient, g_recipient),
    }

def _prewarm_tuple_bertscore(pred_tuples, gold_tuples):
    """Score every action and object pair across the tuple grid in one batch.

    tuple_similarity / tuple_slot_scores call bertscore_similarity per pair, but
    those calls hit _BERTSCORE_CACHE. By batching all (action, action) and
    (object, object) pairs up front, the whole tuple grid is resolved with a
    single GPU call instead of one call per cell.
    """
    pairs = []
    for p in pred_tuples:
        for g in gold_tuples:
            pairs.append((p[1], g[1]))  # action vs action
            pairs.append((p[2], g[2]))  # object vs object
    if pairs:
        bertscore_similarity_batch(pairs)  # populates _BERTSCORE_CACHE

def evaluate_tuples(pred_dialogue, gold_dialogue, threshold=0.70):
    pred_tuples = flatten_semantic_tuples(pred_dialogue)
    gold_tuples = flatten_semantic_tuples(gold_dialogue)

    # Batch every action/object BERTScore pair before scoring the grid.
    _prewarm_tuple_bertscore(pred_tuples, gold_tuples)

    result = bipartite_match(pred_tuples, gold_tuples, tuple_similarity, threshold=threshold)
    result["n_pred_tuples"] = len(pred_tuples)
    result["n_gold_tuples"] = len(gold_tuples)
    for m in result["matches"]:
        m["slot_scores"] = tuple_slot_scores(m["pred_item"], m["gold_item"])
    return result

tuple_eval = evaluate_tuples(pred_data[0], gold_data[0], threshold=0.60)
tuple_eval["precision"], tuple_eval["recall"], tuple_eval["f1"], tuple_eval["matches"]


(1.0,
 0.5,
 0.6666666666666666,
 [{'pred_index': 0,
   'gold_index': 0,
   'pred_item': ('mark', 'was dishonest about his age', 'age', 'anne'),
   'gold_item': ('mark', 'lied about age', 'age', 'anne'),
   'similarity': 0.7914129704236984,
   'slot_scores': {'actor': 1.0,
    'action': 0.4040370583534241,
    'object': 1.0,
    'recipient': 1.0}}])

## 11. Speech-Act Category Mapping

Speech-act tags from Agent 1 and from the gold annotations are free text, so the
same act can carry different labels (e.g. gold `disclosure` vs. predicted
`statement`). Strict string matching would count those as a miss.

To allow a relaxed comparison, each raw tag is mapped to one of six fixed parent
categories with a hand-written dictionary, `SPEECH_ACT_MAP`:

    assertive, directive, question, responsive, expressive, commissive

This is fast, fully transparent, and easy to audit -- no model is loaded. The
tag vocabulary is small, so the dictionary is short.

**Workflow:** run the tag-inventory cell below first. It prints every distinct
`speech_act` tag across the gold and prediction files and flags any tag that is
not yet in `SPEECH_ACT_MAP`. Add any missing tags to the dictionary, then
re-run. Unmapped tags fall back to their raw normalized form, so the relaxed
score still works -- it just will not merge that tag with synonyms.


In [14]:
# Parent speech-act categories.
SPEECH_ACT_CATEGORIES = [
    "assertive",   # stating, informing, explaining, reporting, claiming, disclosing
    "directive",   # requesting, suggesting, advising, instructing, commanding
    "question",    # asking for information or clarification
    "responsive",  # answering, agreeing, refusing, accepting, rejecting
    "expressive",  # thanking, apologizing, complaining, reacting emotionally
    "commissive",  # promising, offering, volunteering, committing to future action
]

# Hand-written map from raw (lowercased) speech-act tags to parent categories.
# Keys must be lowercase; tags are normalized with normalize_entity before lookup.
# Run the tag-inventory cell to find any tags missing here.
SPEECH_ACT_MAP = {
    # --- assertive ---------------------------------------------------------
    "statement": "assertive",
    "assertion": "assertive",
    "assertive": "assertive",
    "claim": "assertive",
    "disclosure": "assertive",
    "disclose": "assertive",
    "inform": "assertive",
    "informing": "assertive",
    "report": "assertive",
    "reporting": "assertive",
    "explanation": "assertive",
    "explain": "assertive",
    "description": "assertive",
    "observation": "assertive",
    "confirmation": "assertive",
    "confirm": "assertive",
    "denial": "assertive",
    "deny": "assertive",
    # --- directive ---------------------------------------------------------
    "directive": "directive",
    "request": "directive",
    "suggestion": "directive",
    "suggest": "directive",
    "advice": "directive",
    "instruction": "directive",
    "command": "directive",
    "order": "directive",
    "proposal": "directive",
    "invitation": "directive",
    "invite": "directive",
    # --- question ----------------------------------------------------------
    "question": "question",
    "query": "question",
    "inquiry": "question",
    "ask": "question",
    "clarification": "question",
    # --- responsive --------------------------------------------------------
    "response": "responsive",
    "responsive": "responsive",
    "answer": "responsive",
    "reply": "responsive",
    "agreement": "responsive",
    "agree": "responsive",
    "acceptance": "responsive",
    "accept": "responsive",
    "rejection": "responsive",
    "reject": "responsive",
    "refusal": "responsive",
    "refuse": "responsive",
    "acknowledgement": "responsive",
    "acknowledgment": "responsive",
    # --- expressive --------------------------------------------------------
    "expressive": "expressive",
    "reaction": "expressive",
    "react": "expressive",
    "exclamation": "expressive",
    "thanks": "expressive",
    "thanking": "expressive",
    "apology": "expressive",
    "apologize": "expressive",
    "complaint": "expressive",
    "complain": "expressive",
    "greeting": "expressive",
    "compliment": "expressive",
    "surprise": "expressive",
    "emotion": "expressive",
    # --- commissive --------------------------------------------------------
    "commissive": "commissive",
    "promise": "commissive",
    "offer": "commissive",
    "commitment": "commissive",
    "commit": "commissive",
    "pledge": "commissive",
    "volunteer": "commissive",
}

def speech_act_category(tag):
    """Map a raw speech-act tag to its parent category.

    Returns the mapped category, or -- if the tag is not in SPEECH_ACT_MAP --
    the tag's own normalized form, so an unmapped tag still matches an identical
    unmapped tag (it just will not merge with synonyms).
    """
    tag_norm = normalize_entity(tag)
    if tag_norm is None:
        return None
    return SPEECH_ACT_MAP.get(tag_norm, tag_norm)

# Quick check on the example tag.
print("disclosure ->", speech_act_category("disclosure"))
print("statement  ->", speech_act_category("statement"))


disclosure -> assertive
statement  -> assertive


In [15]:
# --- Tag inventory --------------------------------------------------------
# Lists every distinct speech_act tag across gold + prediction data and flags
# any tag missing from SPEECH_ACT_MAP. Run this after loading/extracting data.
# If a tag shows "MISSING", add it to SPEECH_ACT_MAP in the cell above.

def speech_act_tag_inventory(*datasets):
    counts = {}
    for data in datasets:
        for dialogue in data:
            for item in get_groundings(dialogue):
                if isinstance(item, dict):
                    tag = normalize_entity(item.get("speech_act"))
                    if tag:
                        counts[tag] = counts.get(tag, 0) + 1

    rows = []
    for tag in sorted(counts):
        mapped = SPEECH_ACT_MAP.get(tag)
        rows.append({
            "tag": tag,
            "count": counts[tag],
            "category": mapped if mapped is not None else "(unmapped -> raw tag)",
            "in_map": mapped is not None,
        })
    inv = pd.DataFrame(rows, columns=["tag", "count", "category", "in_map"])

    missing = inv[~inv["in_map"]]["tag"].tolist()
    print(f"{len(inv)} distinct speech-act tags; {len(missing)} not in SPEECH_ACT_MAP.")
    if missing:
        print("MISSING (add these to SPEECH_ACT_MAP):", missing)
    else:
        print("All tags are mapped.")
    return inv

# Uses the extracted gold_data / pred_data. If they are not built yet, run the
# extraction cell first.
try:
    speech_act_inventory = speech_act_tag_inventory(gold_data, pred_data)
    display(speech_act_inventory)
except NameError:
    print("gold_data / pred_data not defined yet -- run the extraction cell, "
          "then re-run this cell.")


3 distinct speech-act tags; 0 not in SPEECH_ACT_MAP.
All tags are mapped.


,tag,count,category,in_map
0,disclosure,1,assertive,True
1,reaction,1,expressive,True
2,statement,1,assertive,True


## 12. Speech Act Evaluation

In [16]:

def speech_act_strict_similarity(pred_act, gold_act):
    """Exact raw-tag match."""
    return 1.0 if normalize_entity(pred_act) == normalize_entity(gold_act) else 0.0

def speech_act_relaxed_similarity(pred_act, gold_act):
    """Match after mapping both tags to their parent category via SPEECH_ACT_MAP."""
    pred_cat = speech_act_category(pred_act)
    gold_cat = speech_act_category(gold_act)
    if pred_cat is None or gold_cat is None:
        return 0.0
    return 1.0 if pred_cat == gold_cat else 0.0

def evaluate_speech_acts(pred_dialogue, gold_dialogue):
    pred_acts = flatten_speech_acts(pred_dialogue)
    gold_acts = flatten_speech_acts(gold_dialogue)

    # threshold=1.0: only exact (strict) / same-category (relaxed) pairs count.
    strict = bipartite_match(pred_acts, gold_acts, speech_act_strict_similarity, threshold=1.0)
    relaxed = bipartite_match(pred_acts, gold_acts, speech_act_relaxed_similarity, threshold=1.0)

    # Record the category each matched tag maps to, for inspection.
    for m in relaxed["matches"]:
        m["pred_category"] = speech_act_category(m["pred_item"])
        m["gold_category"] = speech_act_category(m["gold_item"])

    return {
        "n_pred_acts": len(pred_acts),
        "n_gold_acts": len(gold_acts),
        "speech_strict_precision": strict["precision"],
        "speech_strict_recall": strict["recall"],
        "speech_strict_f1": strict["f1"],
        "speech_relaxed_precision": relaxed["precision"],
        "speech_relaxed_recall": relaxed["recall"],
        "speech_relaxed_f1": relaxed["f1"],
        "strict_matches": strict["matches"],
        "relaxed_matches": relaxed["matches"],
    }

evaluate_speech_acts(pred_data[0], gold_data[0])


{'n_pred_acts': 1,
 'n_gold_acts': 2,
 'speech_strict_precision': 0.0,
 'speech_strict_recall': 0.0,
 'speech_strict_f1': 0.0,
 'speech_relaxed_precision': 1.0,
 'speech_relaxed_recall': 0.5,
 'speech_relaxed_f1': 0.6666666666666666,
 'strict_matches': [],
 'relaxed_matches': [{'pred_index': 0,
   'gold_index': 0,
   'pred_item': 'statement',
   'gold_item': 'disclosure',
   'similarity': 1.0,
   'pred_category': 'assertive',
   'gold_category': 'assertive'}]}

## 13. Final Outcome Evaluation

In [17]:
def evaluate_final_outcome(pred_dialogue, gold_dialogue):
    pred_outcome = pred_dialogue.get("final_outcome")
    gold_outcome = gold_dialogue.get("final_outcome")
    # Prewarm both NLI directions in one batched call before scoring.
    nli_entailment_prob_batch([(pred_outcome, gold_outcome),
                               (gold_outcome, pred_outcome)])
    return {
        "final_outcome_nli_similarity": bidirectional_nli_similarity(pred_outcome, gold_outcome),
        "pred_final_outcome": pred_outcome,
        "gold_final_outcome": gold_outcome,
    }

evaluate_final_outcome(pred_data[0], gold_data[0])


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/738M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: cross-encoder/nli-deberta-v3-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

{'final_outcome_nli_similarity': 0.5071822721511126,
 'pred_final_outcome': 'Mark was dishonest with Anne about his age; he is 40, not 30.',
 'gold_final_outcome': 'Mark lied to Anne about his age and is actually 40.'}

## 14. Evaluate One Dialogue

In [32]:

def evaluate_dialogue(pred_dialogue, gold_dialogue, meaning_threshold=None, tuple_threshold=0.70):

    participants = evaluate_participants(pred_dialogue, gold_dialogue)
    meanings = evaluate_intended_meanings(pred_dialogue, gold_dialogue, threshold=meaning_threshold)
    tuples = evaluate_tuples(pred_dialogue, gold_dialogue, threshold=tuple_threshold)
    speech = evaluate_speech_acts(pred_dialogue, gold_dialogue)
    outcome = evaluate_final_outcome(pred_dialogue, gold_dialogue)

    bs = meanings["bertscore"]
    return {
        "test_index": get_test_id(gold_dialogue),

        **participants,

        "n_pred_meanings": meanings["n_pred_meanings"],
        "n_gold_meanings": meanings["n_gold_meanings"],
        "meaning_bertscore_precision": bs["precision"],
        "meaning_bertscore_recall": bs["recall"],
        "meaning_bertscore_f1": bs["f1"],

        "tuple_precision": tuples["precision"],
        "tuple_recall": tuples["recall"],
        "tuple_f1": tuples["f1"],
        "tuple_avg_similarity": tuples["avg_matched_similarity"],
        "n_pred_tuples": tuples["n_pred_tuples"],
        "n_gold_tuples": tuples["n_gold_tuples"],

        "speech_strict_f1": speech["speech_strict_f1"],
        "speech_relaxed_f1": speech["speech_relaxed_f1"],
        "n_pred_speech_acts": speech["n_pred_acts"],
        "n_gold_speech_acts": speech["n_gold_acts"],

        "final_outcome_nli_similarity": outcome["final_outcome_nli_similarity"],

        # Detail payloads (kept out of the flat DataFrame).
        "meaning_bertscore_pred_best_matches": bs["pred_best_matches"],
        "meaning_bertscore_gold_best_matches": bs["gold_best_matches"],
        "tuple_matches": tuples["matches"],
        "speech_matches_relaxed": speech["relaxed_matches"],
        "pred_final_outcome": outcome["pred_final_outcome"],
        "gold_final_outcome": outcome["gold_final_outcome"],
    }

one_result = evaluate_dialogue(pred_data[0], gold_data[0], meaning_threshold=None, tuple_threshold=0.60)
one_result


{'test_index': 23,
 'participant_precision': 1.0,
 'participant_recall': 1.0,
 'participant_f1': 1.0,
 'participant_exact_match': True,
 'n_pred_meanings': 3,
 'n_gold_meanings': 1,
 'meaning_bertscore_precision': 0.6496970057487488,
 'meaning_bertscore_recall': 0.8956152200698853,
 'meaning_bertscore_f1': 0.7530886212644298,
 'tuple_precision': 0.3333333333333333,
 'tuple_recall': 1.0,
 'tuple_f1': 0.5,
 'tuple_avg_similarity': 1.0,
 'n_pred_tuples': 3,
 'n_gold_tuples': 1,
 'speech_strict_f1': 0.0,
 'speech_relaxed_f1': 0.5,
 'n_pred_speech_acts': 3,
 'n_gold_speech_acts': 1,
 'final_outcome_nli_similarity': 0.4969112472899724,
 'meaning_bertscore_pred_best_matches': [{'pred_index': 0,
   'gold_index': 0,
   'pred_item': 'Anne reveals that Mark lied about his age.',
   'gold_item': 'Anne confirms that Mark lied to her about his age.',
   'similarity': 0.8956152200698853},
  {'pred_index': 1,
   'gold_index': 0,
   'pred_item': 'Irene questions the significance of the lie.',
   'gold_

## Optional: Preload BERTScorer

Run this cell once before full evaluation if you want the model-loading step to happen upfront instead of during the first comparison.


In [19]:
# Optional prewarm
get_bertscorer()


BERTScorer(hash=roberta-large_L17_no-idf_version=0.3.12(hug_trans=5.0.0)-rescaled, batch_size=64, nthreads=4)

## 15. Evaluate Full Dataset

This cell now handles both input formats:

1. gold/pred files that are already in evaluator format with `test_index`, `participants`, `semantic_grounding`, and `final_outcome` at the top level;
2. raw pipeline output where `test_index` is outside `agent1_semantic_representation`.

The diagnostics below print extraction counts and `test_index` overlap before running the full metrics.


In [33]:
def is_evaluator_format(item):
    """Return True if item already looks like the evaluator's expected structure."""
    return (
        isinstance(item, dict)
        and "test_index" in item
        and "participants" in item
        and "semantic_grounding" in item
        and "final_outcome" in item
    )

def extract_agent1_structures(raw_data):
    """Extract Agent 1 structures from raw pipeline output or already-extracted data.

    Handles both formats:

    1. Already in evaluator format:
       {"test_index": 23, "participants": [...],
        "semantic_grounding": [...], "final_outcome": "..."}

    2. Raw pipeline format:
       {"test_index": 23, "agent1_semantic_representation": {...}}

    In the raw format, test_index usually sits OUTSIDE
    agent1_semantic_representation, but some pipeline variants nest it inside.
    This function looks in both places.
    """
    extracted = []
    skipped = []

    for i, item in enumerate(raw_data):
        if not isinstance(item, dict):
            skipped.append((i, "item is not a dict"))
            continue

        # Case 1: already extracted / evaluator-ready.
        if is_evaluator_format(item):
            extracted.append(dict(item))
            continue

        # Case 2: raw pipeline output with nested Agent 1 representation.
        agent1 = item.get("agent1_semantic_representation")
        if agent1 is None:
            agent1 = item.get("agent1_semantic_representation_json")

        if isinstance(agent1, str):
            try:
                agent1 = json.loads(agent1)
            except Exception as e:
                skipped.append((i, f"failed to parse agent1 JSON string: {e}"))
                continue

        if not isinstance(agent1, dict):
            skipped.append((i, "no valid agent1_semantic_representation"))
            continue

        # test_index may live on the outer object OR inside agent1.
        test_index = item.get("test_index")
        if test_index is None:
            test_index = agent1.get("test_index")
        if test_index is None:
            skipped.append((i, "no test_index on outer object or inside agent1"))
            continue

        structure = {
            "test_index": test_index,
            "participants": agent1.get("participants", []),
            "semantic_grounding": agent1.get("semantic_grounding", []),
            "final_outcome": agent1.get("final_outcome", ""),
        }
        extracted.append(structure)

    print(f"Extracted {len(extracted)} Agent 1 structures from {len(raw_data)} records.")
    if skipped:
        print(f"Skipped {len(skipped)} records. First skipped examples:", skipped[:5])

    return extracted


In [34]:
def index_by_test_index(data, name="data"):
    """Index records by test_index key, keeping the first of any duplicates.

    """
    indexed = {}
    dups, missing = 0, 0
    for d in data:
        key = d.get("test_index")
        if key is None:
            missing += 1
        elif key in indexed:
            dups += 1          # keep the first occurrence
        else:
            indexed[key] = d

    return indexed

def evaluate_dataset(pred_data, gold_data, meaning_threshold=None, tuple_threshold=0.70):
    pred_by_id = index_by_test_index(pred_data, name="pred_data")
    gold_by_id = index_by_test_index(gold_data, name="gold_data")

    rows = []
    details = []
    missing_ids = []

    print("gold indexed:", len(gold_by_id))
    print("pred indexed:", len(pred_by_id))
    print("overlap:", len(set(gold_by_id.keys()) & set(pred_by_id.keys())))

    for test_index_key, gold_dialogue in gold_by_id.items():
        pred_dialogue = pred_by_id.get(test_index_key)

        result = evaluate_dialogue(pred_dialogue, gold_dialogue, meaning_threshold, tuple_threshold)
        details.append(result)

        row = {k: v for k, v in result.items() if not isinstance(v, (list, dict))}
        row["test_index_key"] = test_index_key
        row["missing_prediction"] = False
        rows.append(row)


    return pd.DataFrame(rows), details

# Extract after loading raw files.
gold_data = extract_agent1_structures(gold_raw)
pred_data = extract_agent1_structures(pred_raw)

print("gold_data length:", len(gold_data))
print("pred_data length:", len(pred_data))
print("gold sample keys:", gold_data[0].keys() if gold_data else "EMPTY")
print("pred sample keys:", pred_data[0].keys() if pred_data else "EMPTY")

# Check ID overlap before running expensive metrics.
gold_ids = {x.get("test_index") for x in gold_data}
pred_ids = {x.get("test_index") for x in pred_data}
print("ID overlap:", len(gold_ids & pred_ids))
print("gold sample IDs:", sorted([x for x in gold_ids if x is not None])[:10])
print("pred sample IDs:", sorted([x for x in pred_ids if x is not None])[:10])

df_results, detailed_results = evaluate_dataset(pred_data, gold_data, meaning_threshold=None, tuple_threshold=0.60)
df_results


Extracted 3 Agent 1 structures from 3 records.
Extracted 3 Agent 1 structures from 3 records.
gold_data length: 3
pred_data length: 3
gold sample keys: dict_keys(['test_index', 'participants', 'semantic_grounding', 'final_outcome'])
pred sample keys: dict_keys(['test_index', 'participants', 'semantic_grounding', 'final_outcome'])
ID overlap: 3
gold sample IDs: [23, 30, 39]
pred sample IDs: [23, 30, 39]
gold indexed: 3
pred indexed: 3
overlap: 3


,test_index,participant_precision,participant_recall,participant_f1,participant_exact_match,n_pred_meanings,n_gold_meanings,meaning_bertscore_precision,meaning_bertscore_recall,meaning_bertscore_f1,...,n_gold_tuples,speech_strict_f1,speech_relaxed_f1,n_pred_speech_acts,n_gold_speech_acts,final_outcome_nli_similarity,pred_final_outcome,gold_final_outcome,test_index_key,missing_prediction
0,23,1.0,1.0,1.0,True,3,1,0.649697,0.895615,0.753089,...,1,0.0,0.5,3,1,0.496911,Anne confirms that Mark lied about being 30 wh...,Mark lied to Anne about his age and is actuall...,23,False
1,30,1.0,1.0,1.0,True,2,2,0.780363,0.780363,0.780363,...,2,1.0,1.0,2,2,0.993215,Carter agreed to lend Mary money after an hour.,Carter will lend Mary some money in an hour.,30,False
2,39,1.0,1.0,1.0,True,3,3,0.598134,0.598134,0.598134,...,3,0.0,0.0,3,3,0.498333,Tina is returning home on a delayed evening fl...,"Tina will take the evening flight home, and Al...",39,False


## 16. Aggregate Results

In [35]:
def aggregate_results(df):
    if df.empty:
        raise ValueError(
            "df_results is empty. Check that gold_data and pred_data were extracted "
            "correctly and that their test_index values overlap. Run the Evaluate "
            "Full Dataset cell and inspect its diagnostics."
        )

    metric_cols = [
        "participant_f1",
        "meaning_bertscore_precision", "meaning_bertscore_recall", "meaning_bertscore_f1",
        "tuple_precision", "tuple_recall", "tuple_f1", "tuple_avg_similarity",
        "speech_strict_f1", "speech_relaxed_f1",
        "final_outcome_nli_similarity",
    ]
    available = [c for c in metric_cols if c in df.columns]
    summary = df[available].mean(numeric_only=True).to_frame("macro_average").T

    summary["schema_valid_rate"] = df["schema_valid"].mean() if "schema_valid" in df.columns else np.nan
    summary["missing_prediction_rate"] = df["missing_prediction"].mean() if "missing_prediction" in df.columns else np.nan
    return summary

summary = aggregate_results(df_results)
summary


,participant_f1,meaning_bertscore_precision,meaning_bertscore_recall,meaning_bertscore_f1,tuple_precision,tuple_recall,tuple_f1,tuple_avg_similarity,speech_strict_f1,speech_relaxed_f1,final_outcome_nli_similarity,schema_valid_rate,missing_prediction_rate
macro_average,1.0,0.676064,0.758037,0.710528,0.777778,1.0,0.833333,0.916649,0.333333,0.5,0.66282,NaN,0.0


## 17. Save Results

In [36]:
output_dir = Path("evaluation_outputs")
output_dir.mkdir(exist_ok=True)

# Important: re-extract and recompute immediately before saving to avoid stale notebook variables.
gold_data = extract_agent1_structures(gold_raw)
pred_data = extract_agent1_structures(pred_raw)

df_results, detailed_results = evaluate_dataset(
    pred_data,
    gold_data,
    meaning_threshold=None,
    tuple_threshold=0.60
)
summary = aggregate_results(df_results)

df_results.to_csv(output_dir / "agent1_structure_eval_by_test_index.csv", index=False)
summary.to_csv(output_dir / "agent1_structure_eval_summary.csv", index=False)
save_json(detailed_results, output_dir / "agent1_structure_eval_details.json")

print("Saved outputs to:", output_dir.resolve())
display(df_results.head())
display(summary)


Extracted 3 Agent 1 structures from 3 records.
Extracted 3 Agent 1 structures from 3 records.
gold indexed: 3
pred indexed: 3
overlap: 3
Saved outputs to: /content/evaluation_outputs


,test_index,participant_precision,participant_recall,participant_f1,participant_exact_match,n_pred_meanings,n_gold_meanings,meaning_bertscore_precision,meaning_bertscore_recall,meaning_bertscore_f1,...,n_gold_tuples,speech_strict_f1,speech_relaxed_f1,n_pred_speech_acts,n_gold_speech_acts,final_outcome_nli_similarity,pred_final_outcome,gold_final_outcome,test_index_key,missing_prediction
0,23,1.0,1.0,1.0,True,3,1,0.649697,0.895615,0.753089,...,1,0.0,0.5,3,1,0.496911,Anne confirms that Mark lied about being 30 wh...,Mark lied to Anne about his age and is actuall...,23,False
1,30,1.0,1.0,1.0,True,2,2,0.780363,0.780363,0.780363,...,2,1.0,1.0,2,2,0.993215,Carter agreed to lend Mary money after an hour.,Carter will lend Mary some money in an hour.,30,False
2,39,1.0,1.0,1.0,True,3,3,0.598134,0.598134,0.598134,...,3,0.0,0.0,3,3,0.498333,Tina is returning home on a delayed evening fl...,"Tina will take the evening flight home, and Al...",39,False


,participant_f1,meaning_bertscore_precision,meaning_bertscore_recall,meaning_bertscore_f1,tuple_precision,tuple_recall,tuple_f1,tuple_avg_similarity,speech_strict_f1,speech_relaxed_f1,final_outcome_nli_similarity,schema_valid_rate,missing_prediction_rate
macro_average,1.0,0.676064,0.758037,0.710528,0.777778,1.0,0.833333,0.916649,0.333333,0.5,0.66282,NaN,0.0
